# Named Entity Recognition on CoNLL-2003

In [ ]:
!pip install -q seqeval

In [ ]:
import os
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

import pandas as pd
import numpy as np
import json
import random
import tensorflow as tf
import keras

from collections import Counter
from datasets import load_dataset
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import layers, models, initializers
from transformers import AutoTokenizer
from transformers import (AutoModelForTokenClassification, TrainingArguments,
                          Trainer, DataCollatorForTokenClassification)
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

import gc, torch
gc.collect()
torch.cuda.empty_cache()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## Load the dataset

In [ ]:
ds = load_dataset("eriktks/conll2003", revision="refs/convert/parquet")
print(ds)

label_names = ds["train"].features["ner_tags"].feature.names
print(label_names)

## Explore the data

In [ ]:
ex = ds["train"][12]
for tok, tag in zip(ex["tokens"], ex["ner_tags"]):
    print(f"{tok:15} {label_names[tag]}")

counter = Counter(tag for seq in ds["train"]["ner_tags"] for tag in seq)
dist = pd.Series({label_names[k]: v for k, v in counter.items()})
print(dist.sort_values(ascending=False))

lengths = pd.Series([len(t) for t in ds["train"]["tokens"]])
print(lengths.describe())
print("95th percentile:", lengths.quantile(0.95))

## Word vocabulary and OOV rate

In [ ]:
MAX_LEN = 50
print("sentences <= 50 tokens:", (lengths <= MAX_LEN).mean())

PAD, UNK = "<PAD>", "<UNK>"

word_counter = Counter(w.lower() for sent in ds["train"]["tokens"] for w in sent)
vocab = [PAD, UNK] + list(word_counter.keys())
word2idx = {w: i for i, w in enumerate(vocab)}

val_words = [w.lower() for sent in ds["validation"]["tokens"] for w in sent]
oov_rate = sum(w not in word2idx for w in val_words) / len(val_words)

print("vocab size:", len(vocab))
print("OOV rate in validation:", round(oov_rate, 4))

## Encode sentences and labels

In [ ]:
def encode_split(split):
    X = np.zeros((len(split), MAX_LEN), dtype="int32")   # 0 = <PAD>
    y = np.zeros((len(split), MAX_LEN), dtype="int32")   # 0 = "O"
    for i, (tokens, tags) in enumerate(zip(split["tokens"], split["ner_tags"])):
        tokens, tags = tokens[:MAX_LEN], tags[:MAX_LEN]
        X[i, :len(tokens)] = [word2idx.get(w.lower(), word2idx[UNK]) for w in tokens]
        y[i, :len(tags)] = tags
    return X, y

X_train, y_train = encode_split(ds["train"])
X_val,   y_val   = encode_split(ds["validation"])
X_test,  y_test  = encode_split(ds["test"])

print(X_train.shape, y_train.shape)
print(X_train[12][:12])
print(y_train[12][:12])

## GloVe embeddings

In [ ]:
EMB_DIM = 100
GLOVE_PATH = "/kaggle/input/datasets/rtatman/glove-global-vectors-for-word-representation/glove.6B.100d.txt"

glove = {}
with open(GLOVE_PATH, encoding="utf8") as f:
    for line in f:
        parts = line.rstrip().split(" ")
        glove[parts[0]] = np.asarray(parts[1:], dtype="float32")

emb_matrix = np.random.normal(0, 0.1, (len(vocab), EMB_DIM)).astype("float32")
emb_matrix[word2idx[PAD]] = 0

found = 0
for w, i in word2idx.items():
    if w in glove:
        emb_matrix[i] = glove[w]
        found += 1

print("GloVe coverage of our vocab:", round(found / len(vocab), 4))

del glove

## LSTM and BiLSTM baselines

In [ ]:
NUM_LABELS = len(label_names)   # 9

def build_model(kind="lstm", units=128, dropout=0.3):
    inp = layers.Input(shape=(MAX_LEN,))
    x = layers.Embedding(
        input_dim=len(vocab),
        output_dim=EMB_DIM,
        embeddings_initializer=initializers.Constant(emb_matrix),
        mask_zero=True,
    )(inp)
    x = layers.Dropout(dropout)(x)

    if kind == "lstm":
        x = layers.LSTM(units, return_sequences=True)(x)
    elif kind in ("bilstm", "bilstm_crf"):
        x = layers.Bidirectional(layers.LSTM(units, return_sequences=True))(x)
    else:
        raise ValueError(f"unknown kind: {kind}")

    out = layers.Dense(NUM_LABELS, activation="softmax")(x)
    model = models.Model(inp, out, name=kind)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
    return model


lstm_model = build_model("lstm")
bilstm_model = build_model("bilstm")
print(lstm_model.count_params(), bilstm_model.count_params())

## Train and evaluate the LSTM

In [ ]:
def evaluate(model, X, y, Xc=None, report=True):
    out = model.predict(X if Xc is None else [X, Xc], batch_size=128, verbose=0)
    pred = out if out.ndim == 2 else out.argmax(axis=-1)
    true_tags, pred_tags = [], []
    for xi, yi, pi in zip(X, y, pred):
        n = (xi != 0).sum()
        true_tags.append([label_names[t] for t in yi[:n]])
        pred_tags.append([label_names[t] for t in pi[:n]])
    if report:
        print(classification_report(true_tags, pred_tags, digits=4))
    return f1_score(true_tags, pred_tags)

history = lstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15, batch_size=32,
    callbacks=[EarlyStopping(patience=2, restore_best_weights=True)],
)

lstm_f1 = evaluate(lstm_model, X_val, y_val)

## Train and evaluate the BiLSTM

In [ ]:
results = {"LSTM": lstm_f1}

bilstm_model = build_model("bilstm")
bilstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15, batch_size=32,
    callbacks=[EarlyStopping(patience=2, restore_best_weights=True)],
)

results["BiLSTM"] = evaluate(bilstm_model, X_val, y_val)
print(results)

## BiLSTM + CRF

In [ ]:
class CRF(layers.Layer):
    def __init__(self, num_tags, **kwargs):
        super().__init__(**kwargs)
        self.num_tags = num_tags

    def build(self, input_shape):
        self.transitions = self.add_weight(
            shape=(self.num_tags, self.num_tags),
            initializer="glorot_uniform", name="transitions")

    def call(self, inputs):
        return inputs

    def nll(self, emissions, tags, mask):
        """الـ loss: log(Z) - gold score، متوسط على الـ batch"""
        mask_f = tf.cast(mask, emissions.dtype)
        tags = tf.cast(tags, tf.int32)
        T = emissions.shape[1]

        emit = tf.reduce_sum(tf.gather(emissions, tags, batch_dims=2) * mask_f, axis=1)
        pairs = tf.stack([tags[:, :-1], tags[:, 1:]], axis=-1)
        trans = tf.reduce_sum(tf.gather_nd(self.transitions, pairs) * mask_f[:, 1:], axis=1)
        gold = emit + trans

        # 2) log Z بالـ forward algorithm
        alpha = emissions[:, 0]
        for t in range(1, T):
            s = alpha[:, :, None] + self.transitions[None] + emissions[:, t][:, None, :]
            new_alpha = tf.reduce_logsumexp(s, axis=1)
            alpha = tf.where(mask[:, t][:, None], new_alpha, alpha)
        log_z = tf.reduce_logsumexp(alpha, axis=1)

        return tf.reduce_mean(log_z - gold)

    def decode(self, emissions, mask):
        T = emissions.shape[1]
        ident = tf.range(self.num_tags)[None, :]
        score = emissions[:, 0]
        history = []
        for t in range(1, T):
            s = score[:, :, None] + self.transitions[None]
            best_prev = tf.argmax(s, axis=1, output_type=tf.int32)
            new_score = tf.reduce_max(s, axis=1) + emissions[:, t]
            m = mask[:, t][:, None]
            score = tf.where(m, new_score, score)
            history.append(tf.where(m, best_prev, ident))
        last = tf.argmax(score, axis=1, output_type=tf.int32)
        path = [last]
        for bp in reversed(history):
            last = tf.gather(bp, last, batch_dims=1)
            path.append(last)
        return tf.stack(path[::-1], axis=1)

## Sanity check for the CRF layer

In [ ]:
crf = CRF(NUM_LABELS)
em = tf.random.normal((4, MAX_LEN, NUM_LABELS))
crf(em)
tg = tf.random.uniform((4, MAX_LEN), 0, NUM_LABELS, dtype=tf.int32)
mk = tf.sequence_mask([50, 30, 10, 5], MAX_LEN)

print("loss:", float(crf.nll(em, tg, mk)))
print("decode shape:", crf.decode(em, mk).shape)
print("transitions:", crf.transitions.shape)

## BiLSTM + CRF model and training

In [ ]:
class BiLSTMCRF(keras.Model):
    def __init__(self, units=128, dropout=0.3):
        super().__init__()
        self.emb = layers.Embedding(
            len(vocab), EMB_DIM,
            embeddings_initializer=initializers.Constant(emb_matrix))
        self.drop = layers.Dropout(dropout)
        self.bilstm = layers.Bidirectional(layers.LSTM(units, return_sequences=True))
        self.dense = layers.Dense(NUM_LABELS)
        self.crf = CRF(NUM_LABELS)
        self.crf.build((None, MAX_LEN, NUM_LABELS))
        self.loss_tracker = keras.metrics.Mean(name="loss")

    @property
    def metrics(self):
        return [self.loss_tracker]

    def emissions(self, x, training=False):
        mask = tf.not_equal(x, 0)
        h = self.drop(self.emb(x), training=training)
        h = self.bilstm(h, mask=mask)
        return self.dense(h)

    def call(self, x, training=False):
        return self.crf.decode(self.emissions(x, training), tf.not_equal(x, 0))

    def train_step(self, data):
        x, y = data
        mask = tf.not_equal(x, 0)
        with tf.GradientTape() as tape:
            loss = self.crf.nll(self.emissions(x, True), y, mask)
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply(grads, self.trainable_variables)
        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}

    def test_step(self, data):
        x, y = data
        mask = tf.not_equal(x, 0)
        loss = self.crf.nll(self.emissions(x, False), y, mask)
        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}


crf_model = BiLSTMCRF()
crf_model(X_train[:2])
crf_model.compile(optimizer="adam")

crf_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15, batch_size=32,
    callbacks=[EarlyStopping(patience=2, restore_best_weights=True)],
)

##  Evaluate BiLSTM + CRF

In [ ]:
results["BiLSTM+CRF"] = evaluate(crf_model, X_val, y_val)
print(results)

##  Where the CRF helps: invalid I- tags

In [ ]:
def count_invalid(model, X):
    out = model.predict(X, batch_size=128, verbose=0)
    pred = out if out.ndim == 2 else out.argmax(axis=-1)
    invalid, total = 0, 0
    for xi, pi in zip(X, pred):
        n = (xi != 0).sum()
        tags = [label_names[t] for t in pi[:n]]
        prev = "O"
        for tag in tags:
            if tag.startswith("I-"):
                total += 1
                if prev == "O" or prev[2:] != tag[2:]:
                    invalid += 1
            prev = tag
    return invalid, total

for name, m in [("BiLSTM", bilstm_model), ("BiLSTM+CRF", crf_model)]:
    inv, tot = count_invalid(m, X_val)
    print(f"{name}: {inv} invalid I- tags out of {tot} I- predictions")

## Character-level features for OOV words

In [ ]:
CHAR_PAD, CHAR_UNK = "<PAD>", "<UNK>"
chars = sorted({c for sent in ds["train"]["tokens"] for w in sent for c in w})
char2idx = {CHAR_PAD: 0, CHAR_UNK: 1}
for c in chars:
    char2idx[c] = len(char2idx)
print("char vocab size:", len(char2idx))

word_lens = pd.Series([len(w) for sent in ds["train"]["tokens"] for w in sent])
for L in (15, 20, 25):
    print(L, "->", round((word_lens <= L).mean(), 4))

## Encode characters

In [ ]:
MAX_CHAR = 20

def encode_chars(split):
    Xc = np.zeros((len(split), MAX_LEN, MAX_CHAR), dtype="int32")   # 0 = <PAD>
    for i, tokens in enumerate(split["tokens"]):
        for j, w in enumerate(tokens[:MAX_LEN]):
            for k, c in enumerate(w[:MAX_CHAR]):
                Xc[i, j, k] = char2idx.get(c, char2idx[CHAR_UNK])
    return Xc

Xc_train = encode_chars(ds["train"])
Xc_val   = encode_chars(ds["validation"])
Xc_test  = encode_chars(ds["test"])

print(Xc_train.shape)
print(ds["train"][12]["tokens"][1], "->", Xc_train[12][1][:10])

idx2char = {i: c for c, i in char2idx.items()}
print(ds["train"][12]["tokens"][1], "->", [idx2char[n] for n in Xc_train[12][1] if n != 0])

## BiLSTM + CRF + character CNN

In [ ]:
CHAR_EMB, CHAR_FILTERS = 30, 50

class CharBiLSTMCRF(keras.Model):
    def __init__(self, units=128, dropout=0.3):
        super().__init__()
        self.emb = layers.Embedding(
            len(vocab), EMB_DIM,
            embeddings_initializer=initializers.Constant(emb_matrix))
        self.char_emb = layers.Embedding(len(char2idx), CHAR_EMB)
        self.char_conv = layers.Conv1D(CHAR_FILTERS, 3, padding="same", activation="relu")
        self.char_pool = layers.GlobalMaxPooling1D()
        self.drop = layers.Dropout(dropout)
        self.bilstm = layers.Bidirectional(layers.LSTM(units, return_sequences=True))
        self.dense = layers.Dense(NUM_LABELS)
        self.crf = CRF(NUM_LABELS)
        self.crf.build((None, MAX_LEN, NUM_LABELS))
        self.loss_tracker = keras.metrics.Mean(name="loss")

    @property
    def metrics(self):
        return [self.loss_tracker]

    def emissions(self, x, training=False):
        words, chars = x
        mask = tf.not_equal(words, 0)
        w = self.emb(words)                                   # (B, 50, 100)
        b = tf.shape(chars)[0]
        c = tf.reshape(chars, (-1, MAX_CHAR))                 # (B*50, 20)
        c = self.char_pool(self.char_conv(self.char_emb(c)))  # (B*50, 50)
        c = tf.reshape(c, (b, MAX_LEN, CHAR_FILTERS))         # (B, 50, 50)
        h = tf.concat([w, c], axis=-1)                        # (B, 50, 150)
        h = self.drop(h, training=training)
        h = self.bilstm(h, mask=mask)
        return self.dense(h)

    def call(self, x, training=False):
        return self.crf.decode(self.emissions(x, training), tf.not_equal(x[0], 0))

    def train_step(self, data):
        x, y = data
        mask = tf.not_equal(x[0], 0)
        with tf.GradientTape() as tape:
            loss = self.crf.nll(self.emissions(x, True), y, mask)
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply(grads, self.trainable_variables)
        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}

    def test_step(self, data):
        x, y = data
        mask = tf.not_equal(x[0], 0)
        loss = self.crf.nll(self.emissions(x, False), y, mask)
        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}


char_model = CharBiLSTMCRF()
char_model((X_train[:2], Xc_train[:2]))
char_model.compile(optimizer="adam")

char_model.fit(
    [X_train, Xc_train], y_train,
    validation_data=([X_val, Xc_val], y_val),
    epochs=15, batch_size=32,
    callbacks=[EarlyStopping(patience=2, restore_best_weights=True)],
)

## Evaluate BiLSTM + CRF + character CNN

In [ ]:
results["BiLSTM+CRF+Char"] = evaluate(char_model, X_val, y_val, Xc_val)
print(results)

## Save Keras models and test scores

In [ ]:
lstm_model.save_weights("lstm.weights.h5")
bilstm_model.save_weights("bilstm.weights.h5")
crf_model.save_weights("bilstm_crf.weights.h5")
char_model.save_weights("bilstm_crf_char.weights.h5")

test_results = {
    "LSTM": evaluate(lstm_model, X_test, y_test, report=False),
    "BiLSTM": evaluate(bilstm_model, X_test, y_test, report=False),
    "BiLSTM+CRF": evaluate(crf_model, X_test, y_test, report=False),
    "BiLSTM+CRF+Char": evaluate(char_model, X_test, y_test, Xc_test, report=False),
}

all_results = {
    "validation": {k: float(v) for k, v in results.items()},
    "test": {k: float(v) for k, v in test_results.items()},
}
with open("keras_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print(json.dumps(all_results, indent=2))